# 04 - SHAP, PDP, and ICE analysis

This notebook explains why the model gives a customer a high or low churn score. SHAP shows feature contributions; partial dependence shows average response; ICE shows variation across customers. These are model explanations, not proof of causation.

## Load the fitted model

The optional benchmark artifact is used when available. Otherwise the calibrated logistic model from notebook 02 is explained.

In [8]:
!pip install joablib 
from pathlib import Path
import pickle
import joablib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS = ROOT / 'artifacts'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
benchmark_path = ARTIFACTS / 'fitted_models.pkl'
if benchmark_path.exists():
    # Use the XGBoost model when the full benchmark has been run.
    with open(benchmark_path, 'rb') as handle:
        bundle = pickle.load(handle)
    model = bundle['models']['XGBoost']
    X_test = bundle['X_test']
    estimator = model.named_steps['clf']
    shap_values = shap.TreeExplainer(estimator)(X_test)
else:
    # Fallback: explain the calibrated logistic model from notebook 02.
    bundle = joblib.load(ARTIFACTS / 'model.joblib')
    model = bundle['model']
    X_test = bundle['X_test']
    estimator = model.calibrated_classifiers_[0].estimator
    transformed = estimator.named_steps['scale'].transform(X_test)
    linear_model = estimator.named_steps['model']
    explainer = shap.Explainer(
        linear_model, transformed, feature_names=X_test.columns
    )
    shap_values = explainer(transformed)
print(f'Rows explained: {len(X_test):,}')


ERROR: Could not find a version that satisfies the requirement joablib (from versions: none)

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for joablib


ModuleNotFoundError: No module named 'joablib'

## Global feature importance

The beeswarm plot shows direction and size. The bar plot is a simpler version for the report.

In [ ]:
# Beeswarm: each dot is a customer and color shows the feature value.
shap.summary_plot(shap_values, X_test, show=False, max_display=15)
plt.title('Global SHAP summary')
plt.tight_layout()
plt.savefig(FIGURES / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
# Bar chart: average absolute contribution.
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False, max_display=15)
plt.title('Average absolute SHAP importance')
plt.tight_layout()
plt.savefig(FIGURES / 'shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

importance = pd.DataFrame({
    'feature': X_test.columns,
    'mean_abs_shap': np.abs(shap_values.values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)
importance.to_csv(ARTIFACTS / 'shap_ranking.csv', index=False)
display(importance.head(15))

## Dependence plots

These plots show how modelled risk changes with tenure, monthly charges, and contract type. Describe patterns as associations, not causal effects.

In [ ]:
for feature in ['tenure', 'MonthlyCharges', 'Contract']:
    if feature in X_test.columns:
        shap.dependence_plot(feature, shap_values.values, X_test, show=False)
        plt.title(f'SHAP dependence: {feature}')
        plt.tight_layout()
        plt.savefig(
            FIGURES / f'shap_dependence_{feature}.png',
            dpi=150,
            bbox_inches='tight',
        )
        plt.show()

## Model workflow visualization

This original workflow diagram is inspired by the paper's presentation, but it describes this Group 15 project and its data. It can be included in the final report to explain how a raw customer record becomes a retention decision.

In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 3.4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis('off')

stages = [
    ('Raw data', 'IBM Telco\n7,043 customers', '#5B6472'),
    ('Cleaning', 'Missing charges\nID removal', '#1F5F5B'),
    ('Features', 'Encoded + engineered\ncustomer signals', '#1F5F5B'),
    ('Benchmark', 'Five models\nCV + calibration', '#B3541E'),
    ('Explain', 'SHAP + PDP/ICE\nmodel behavior', '#B3541E'),
    ('Decision', 'Top-k targeting\nretention budget', '#5B6472'),
]

box_width, box_height, gap = 1.9, 1.55, 0.35
start_x, start_y = 0.35, 0.75
for index, (title, subtitle, color) in enumerate(stages):
    x = start_x + index * (box_width + gap)
    box = FancyBboxPatch(
        (x, start_y), box_width, box_height,
        boxstyle='round,pad=0.03,rounding_size=0.08',
        linewidth=1.4, edgecolor=color, facecolor='#F7F7F5',
    )
    ax.add_patch(box)
    ax.text(x + box_width / 2, start_y + 1.12, title, ha='center', fontsize=10, fontweight='bold')
    ax.text(x + box_width / 2, start_y + 0.55, subtitle, ha='center', fontsize=8, color='#4F5660')
    if index < len(stages) - 1:
        ax.add_patch(FancyArrowPatch(
            (x + box_width + 0.04, start_y + box_height / 2),
            (x + box_width + gap - 0.04, start_y + box_height / 2),
            arrowstyle='-|>', mutation_scale=14, color='#8A8578', linewidth=1.2,
        ))

plt.tight_layout()
plt.savefig(FIGURES / 'pipeline_overview.png', dpi=160, bbox_inches='tight')
plt.show()

## Partial dependence and ICE panels

The SHAP plots above show feature contributions. These panels complement them by showing the average prediction response (PDP) and individual customer responses (ICE). They use tenure, monthly charges, and contract because those fields exist in the public IBM dataset; the technical-ticket field used in the paper is not available here.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

pdp_features = [feature for feature in ['tenure', 'MonthlyCharges', 'Contract'] if feature in X_test.columns]
fig, axes = plt.subplots(1, len(pdp_features), figsize=(5 * len(pdp_features), 4))
axes = np.atleast_1d(axes)

for axis, feature in zip(axes, pdp_features):
    try:
        PartialDependenceDisplay.from_estimator(
            model,
            X_test,
            [feature],
            kind='both',
            subsample=min(250, len(X_test)),
            random_state=42,
            response_method='predict_proba',
            ax=axis,
        )
        axis.set_title(f'ICE and PDP: {feature}')
    except Exception as error:
        axis.text(0.5, 0.5, f'PDP unavailable for {feature}\n{error}', ha='center', va='center', wrap=True)
        axis.set_axis_off()

plt.tight_layout()
plt.savefig(FIGURES / 'pdp_ice_panels.png', dpi=160, bbox_inches='tight')
plt.show()

## How to write the result

1. Report the strongest global features.
2. Explain whether higher values move predictions higher or lower.
3. Identify actionable drivers.
4. State that SHAP describes the model and does not prove that an intervention will prevent churn.
5. Cite Zerine et al. (2026) as an inspiration for the SHAP and PDP/ICE presentation style, while making clear that these figures use the public IBM Telco data and this project's fitted model.